# FJ Parameters

Heatmaps of the long-run average opinion shift compared with the vanilla Friedkin-Johnsen model, across the (`community_ratio`, `stubbornness_mean`) parameter grid (Section 3.2). Reads from `outputs/fj_params/`.

In [1]:
import os
os.chdir("../")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import glob
from src import utils

sns.set_theme(context='paper', style='ticks', font_scale=1)

os.environ['PATH'] = f"{os.path.expanduser('~/.TinyTeX/bin/x86_64-linux')}:{os.environ['PATH']}"

In [ ]:
name = "fj_params"
exp_name = "fj_params"
width_pt = 469

dataset = "semeval"
task = "improvement"
model_name = "google/gemma-3-12b-it"
quantification_method = "centroid"

community_ratios = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
stubbornness_means = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

target_pct = 0.6

model_str = model_name.replace("/", "_")
results_dir = f"outputs/{name}"


def result_path(network, topic, community_ratio, stubbornness_mean, transformed_pct):
    return (
        f"{results_dir}/{exp_name}__network={network}"
        f"__community_ratio={community_ratio}"
        f"__stubbornness_mean={stubbornness_mean}"
        f"__transformed_pct={transformed_pct}"
        f"__dataset={dataset}__task={task}__topic={topic}"
        f"__model={model_str}__quantification_method={quantification_method}.json"
    )

## Effect of AI mediation on long-run average opinion

In [ ]:
dynamics_pattern = (
    f"{results_dir}/{exp_name}__network=*"
    f"__community_ratio=*"
    f"__stubbornness_mean=*"
    f"__transformed_pct={target_pct}"
    f"__dataset={dataset}__task={task}__topic=*"
    f"__model={model_str}__quantification_method={quantification_method}.json"
)
network_topic_pairs = set()
for path in glob.glob(dynamics_pattern):
    basename = os.path.basename(path).replace(".json", "")
    parts = dict(p.split("=", 1) for p in basename.split("__")[1:])
    network_topic_pairs.add((parts["network"], parts["topic"]))
network_topic_pairs = sorted(network_topic_pairs)
print(f"Found {len(network_topic_pairs)} (network, topic) pairs: {network_topic_pairs}")


def equilibrium_mean(network, topic, community_ratio, stubbornness_mean, transformed_pct):
    try:
        with open(result_path(network, topic, community_ratio, stubbornness_mean, transformed_pct)) as f:
            data = json.load(f)
    except FileNotFoundError:
        return np.nan
    return np.mean([
        r["internal_trajectory"][-1]["overall_mean"]
        for r in data["seed_results"]
    ])


utils.latexify()

for nw, tp in network_topic_pairs:
    diff = np.full((len(community_ratios), len(stubbornness_means)), np.nan)
    for i, cr in enumerate(community_ratios):
        for j, sm in enumerate(stubbornness_means):
            diff[i, j] = (
                equilibrium_mean(nw, tp, cr, sm, target_pct)
                - equilibrium_mean(nw, tp, cr, sm, 0)
            )

    n_missing = int(np.isnan(diff).sum())
    if n_missing > 0:
        print(f"Warning [{nw}, {tp}]: {n_missing}/{diff.size} cells missing; rendering NaN as blank.")
    if np.all(np.isnan(diff)):
        continue

    fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    vmax = np.nanmax(np.abs(diff))
    sns.heatmap(
        diff,
        xticklabels=stubbornness_means,
        yticklabels=[f"{cr * 100:.0f}\\%" for cr in community_ratios],
        cmap="icefire",
        center=0,
        vmin=-vmax,
        vmax=vmax,
        cbar_kws={"label": "Long-run average\nopinion shift", "format": "%.2f"},
        ax=ax,
    )
    ax.invert_yaxis()
    ax.set_xlabel("Average stubbornness")
    ax.set_ylabel("Innate opinions in favor")

    fig.tight_layout()
    fig.savefig(
        f"figures/{name}__heatmap__{model_str}__network={nw}__dataset={dataset}__task={task}__topic={tp}"
        f"__target_pct={target_pct}.pdf",
        dpi=300,
    )
    plt.close()

## Same plot as before but with an annotation to highlight the maximum bias amplification

In [ ]:
nw, tp = "twitter", "abortion"


def expressed_initial_mean(network, topic, community_ratio, stubbornness_mean, transformed_pct):
    try:
        with open(result_path(network, topic, community_ratio, stubbornness_mean, transformed_pct)) as f:
            data = json.load(f)
    except FileNotFoundError:
        return np.nan
    return np.mean([
        r["expressed_trajectory"][0]["overall_mean"]
        for r in data["seed_results"]
    ])


diff = np.full((len(community_ratios), len(stubbornness_means)), np.nan)
one_step_bias = np.full((len(community_ratios), len(stubbornness_means)), np.nan)
for i, cr in enumerate(community_ratios):
    for j, sm in enumerate(stubbornness_means):
        diff[i, j] = (
            equilibrium_mean(nw, tp, cr, sm, target_pct)
            - equilibrium_mean(nw, tp, cr, sm, 0)
        )
        one_step_bias[i, j] = (
            expressed_initial_mean(nw, tp, cr, sm, target_pct)
            - expressed_initial_mean(nw, tp, cr, sm, 0)
        )

i_max, j_max = np.unravel_index(np.nanargmax(np.abs(diff)), diff.shape)
max_shift = diff[i_max, j_max]
bias_at_max = one_step_bias[i_max, j_max]
ratio = max_shift / bias_at_max
print(f"Max-|shift| cell: community_ratio={community_ratios[i_max]}, stubbornness_mean={stubbornness_means[j_max]}")
print(f"  equilibrium shift = {max_shift:.4f}")
print(f"  one-step expression bias = {bias_at_max:.4f}")
print(f"  ratio (shift / bias) = {ratio:.3f}")

utils.latexify()

fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

vmax = np.nanmax(np.abs(diff))
sns.heatmap(
    diff,
    xticklabels=stubbornness_means,
    yticklabels=[f"{cr * 100:.0f}\\%" for cr in community_ratios],
    cmap="icefire",
    center=0,
    vmin=-vmax,
    vmax=vmax,
    cbar_kws={"label": "Long-run average\nopinion shift", "format": "%.2f"},
    ax=ax,
)
ax.invert_yaxis()
ax.set_xlabel("Average stubbornness")
ax.set_ylabel("Innate opinions in favor")

text_x, text_y = 5.2, 1
text_left_offset = 3.2
ax.text(
    text_x, text_y,
    rf"Bias amplification: $\times {ratio:.1f}$",
    ha="center", va="center", fontsize=10,
    bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="white", linewidth=0.8),
)

cell_row, cell_col = 3, 0
cell_x = cell_col + 0.5
cell_y = cell_row + 0.5

angle_deg = 75
angle_rad = np.radians(angle_deg)
dy = text_y - cell_y
elbow_x = cell_x + abs(dy) / np.tan(angle_rad)
elbow_y = text_y

overlap = 0.05

ax.plot([cell_x, elbow_x + overlap], [cell_y, elbow_y], color="white", lw=1.2, zorder=5)
ax.annotate("", xy=(text_x - text_left_offset, text_y), xytext=(elbow_x - overlap, elbow_y),
            arrowprops=dict(arrowstyle="-", color="white", lw=1.2, mutation_scale=10))

fig.tight_layout()
fig.savefig(
    f"figures/{name}__heatmap_with_ratio__{model_str}__network={nw}__dataset={dataset}__task={task}__topic={tp}"
    f"__target_pct={target_pct}.pdf",
    dpi=300,
)
plt.show()

In [6]:
amplification = diff / one_step_bias
df = pd.DataFrame(
    amplification,
    index=[f"{cr * 100:.0f}%" for cr in community_ratios],
    columns=[f"{sm:.1f}" for sm in stubbornness_means],
)
df.index.name = "community_ratio"
df.columns.name = "stubbornness_mean"
print(f"Bias amplification (equilibrium shift / one-step bias) for network={nw}, topic={tp}, target_pct={target_pct}")
print(df.round(2).to_string())

Bias amplification (equilibrium shift / one-step bias) for network=twitter, topic=abortion, target_pct=0.6
stubbornness_mean   0.1   0.2   0.3   0.4   0.5   0.6   0.7   0.8  0.9
community_ratio                                                       
10%                2.38  1.33  0.98  0.85  0.72  0.54  0.37  0.22  0.1
20%                1.43  0.77  0.82  0.89  0.76  0.57  0.39  0.23  0.1
30%                1.83  3.37  1.56  1.15  0.84  0.59  0.40  0.23  0.1
40%                9.22  6.57  3.44  1.46  0.94  0.62  0.40  0.23  0.1
50%                7.85  5.50  3.58  1.72  1.00  0.63  0.40  0.23  0.1
60%                6.64  4.56  3.12  1.80  1.01  0.63  0.40  0.23  0.1
70%                5.60  3.73  2.60  1.66  0.98  0.61  0.39  0.23  0.1
80%                4.75  3.05  2.11  1.42  0.90  0.58  0.37  0.22  0.1
90%                4.05  2.47  1.68  1.16  0.79  0.53  0.35  0.21  0.1
